In [1]:
import rasterio
import numpy as np
from rasterio.enums import Resampling
from scipy.spatial import cKDTree

In [2]:
import rasterio

tif_file = r"./data/forest_supply/ESACCI-BIOMASS-L4-AGB-MERGED-1000m-fv6.0.tif"

with rasterio.open(tif_file) as src:
    print("CRS:", src.crs)
    print("Resolution:", src.res)
    print("Width:", src.width)
    print("Height:", src.height)
    print("Bounds:", src.bounds)

CRS: EPSG:4326
Resolution: (0.01, 0.01)
Width: 36000
Height: 18000
Bounds: BoundingBox(left=-180.0, bottom=-90.0, right=180.0, top=90.0)


# Write climate zone types into tif file

In [3]:
# -------------------------
# File paths
# -------------------------
biomass_path = "./data/forest_supply/AGB_2022_5km.tif"
climate_path = "./data/Climate_Zone/IPCC_Climate_Zones_Map_raster/Raster/ipcc_climate_1985-2015.tif"
output_path = "./data/forest_supply/AGB_2022_5km_climate.tif"

# -------------------------
# Read biomass raster
# -------------------------
with rasterio.open(biomass_path) as src:
    biomass_data = src.read(1)
    transform = src.transform
    height = src.height
    width = src.width
    crs = src.crs

# -------------------------
# Read and resample climate raster to biomass grid
# -------------------------
with rasterio.open(climate_path) as src:
    climate_data = src.read(
        1,
        out_shape=(height, width),
        resampling=Resampling.nearest  # preserve classification values
    )

# -------------------------
# Find cells where biomass is not NaN
# -------------------------
rows, cols = np.where(~np.isnan(biomass_data))

# -------------------------
# Create new climate zone array
# -------------------------
climate_filled = np.full_like(biomass_data, np.nan, dtype=np.float32)

# Prioritize overlapping cells
climate_filled[rows, cols] = climate_data[rows, cols]

# -------------------------
# Fill remaining NaN cells using nearest neighbor
# -------------------------
nan_rows, nan_cols = np.where(~np.isnan(biomass_data) & np.isnan(climate_filled))
if len(nan_rows) > 0:
    xs, ys = rasterio.transform.xy(transform, nan_rows, nan_cols)
    points = np.column_stack([xs, ys])

    valid_rows, valid_cols = np.where(~np.isnan(climate_data))
    valid_xs, valid_ys = rasterio.transform.xy(transform, valid_rows, valid_cols)
    valid_points = np.column_stack([valid_xs, valid_ys])
    valid_values = climate_data[valid_rows, valid_cols]

    tree = cKDTree(valid_points)
    dist, idx = tree.query(points)

    for r, c, i in zip(nan_rows, nan_cols, idx):
        climate_filled[r, c] = valid_values[i]

# -------------------------
# Save new two-band tif
# -------------------------
with rasterio.open(
    output_path,
    'w',
    driver='GTiff',
    height=height,
    width=width,
    count=2,  # band1: biomass, band2: filled climate zone
    dtype='float32',
    crs=crs,
    transform=transform
) as dst:
    dst.write(biomass_data, 1)      # band1: original biomass
    dst.write(climate_filled, 2)    # band2: filled climate zone

print("Done! Output path:", output_path)

Done! Output path: ./data/forest_supply/AGB_2022_5km_climate.tif


### Set BCEFs (biomass conversion and expansion factor applicable to growing stock; transforms merchantable volume of growing stock into above-ground biomass) and Residue production per wood production (%) according to climate zone; then calculate available residue.

In [4]:
import rasterio
import numpy as np

In [5]:
# RTWR unit is %, BCEF unit is tonnes d.m. m-3; wood density is 0.58 ton/m3; average energy content of logging residue is 19.8 MJ/kg;
# Above-ground biomass is tons/ha
# File paths and parameters
# -----------------------------
input_tif = "./data/forest_supply/AGB_2022_5km_climate.tif"
output_tif = "./data/forest_supply/avaliable_residue_GJ_per_ha.tif"

wood_density = 0.58       # ton/m3
energy_content = 19.8     # MJ/kg = GJ/ton
fraction = 0.25           # Available fraction (25%)

# -----------------------------
# Classification lookup table
# -----------------------------
mapping = {
    1: {"RTWR": 52, "BCEF": 0.8},
    2: {"RTWR": 52, "BCEF": 0.8},
    3: {"RTWR": 52, "BCEF": 0.8},
    4: {"RTWR": 52, "BCEF": 0.8},
    5: {"RTWR": 52, "BCEF": 1.4},
    6: {"RTWR": 52, "BCEF": 1.4},
    7: {"RTWR": 63, "BCEF": 1.4},
    8: {"RTWR": 63, "BCEF": 1.4},
    9: {"RTWR": 78, "BCEF": 0.62},
    10: {"RTWR": 78, "BCEF": 0.62},
    11: {"RTWR": 78, "BCEF": 0.62},
    12: {"RTWR": 78, "BCEF": 0.62},
}

# -----------------------------
# Read raster
# -----------------------------
with rasterio.open(input_tif) as src:
    biomass = src.read(1)   # band1: biomass
    classes = src.read(2)   # band2: class
    profile = src.profile   # preserve spatial info for later use

# -----------------------------
# Initialize RTWR and BCEF arrays
# -----------------------------
RTWR = np.full(classes.shape, np.nan, dtype=np.float32)
BCEF = np.full(classes.shape, np.nan, dtype=np.float32)

# -----------------------------
# Assign values by class
# -----------------------------
for class_val, vals in mapping.items():
    mask = classes == class_val
    RTWR[mask] = vals["RTWR"]
    BCEF[mask] = vals["BCEF"]

# Set cells with class outside 1–12 range to NaN
invalid_mask = ~np.isin(classes, list(mapping.keys()))
RTWR[invalid_mask] = np.nan
BCEF[invalid_mask] = np.nan

# -----------------------------
# Calculate available residue biomass energy
# -----------------------------
# biomass / BCEF converts merchantable biomass to merchantable wood volume
# × wood_density: convert volume to mass
# × RTWR: wood removal to residue ratio
# × fraction: available fraction (25%)
# × energy_content: convert to energy (GJ/ton)

with np.errstate(divide='ignore', invalid='ignore'):
    avaliable_residue = (biomass / BCEF) * wood_density * (RTWR/100) * fraction * energy_content #unit is GJ/ha

# -----------------------------
# Set invalid cells to NaN
# -----------------------------
avaliable_residue[np.isnan(BCEF)] = np.nan
avaliable_residue[BCEF == 0] = np.nan

profile.update(
    dtype=rasterio.float32,
    count=1,
    nodata=np.nan,
    compress='lzw'  # compress to save space, optional
)

with rasterio.open(output_tif, 'w', **profile) as dst:
    dst.write(avaliable_residue.astype(np.float32), 1)

print(f"Done: calculation complete, result saved to:\n{output_tif}")

Done: calculation complete, result saved to:
./data/forest_supply/avaliable_residue_GJ_per_ha.tif


### Previous step gives residue in GJ/ha per 25 km2; this step uses the plantation forest layer to count how many 250m cells exist in each 5 km cell, to derive plantation forest area

In [6]:
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling
from rasterio.windows import Window
import rioxarray
import os

In [7]:
import warnings
warnings.filterwarnings("ignore", category=rasterio.errors.RasterioDeprecationWarning)

In [8]:
# -----------------------------
# Parameters
# -----------------------------
input_raster = r"./data/biomass_supply/forest_management_map/2020/plantation_forest.tif"
output_raster = r"./data/biomass_supply/forest_management_map/2020/plantation_forest_agg20.tif"
factor = 20

# -----------------------------
# Open large file (using Dask chunks)
# -----------------------------
# Chunks can be set smaller to save memory (e.g. 4096 or 2048)
da = rioxarray.open_rasterio(input_raster, masked=True, chunks={"x": 4096, "y": 4096})

print(f"Original size: {da.rio.width} × {da.rio.height}")
print(f"Original resolution: {da.rio.resolution()}")

# -----------------------------
# Aggregate (count occurrences of 1 in each coarse cell by block)
# -----------------------------
agg = da.coarsen(x=factor, y=factor, boundary="pad").sum()

# Convert to int32 to save space
agg = agg.astype("int32")

# Preserve CRS
agg.rio.write_crs(da.rio.crs, inplace=True)

# -----------------------------
# Write output (using Dask delayed compute)
# -----------------------------
os.makedirs(os.path.dirname(output_raster), exist_ok=True)

# compute=True will compute block by block when writing
agg.rio.to_raster(output_raster, compute=True)

print(f"Done: aggregation complete! Output: {output_raster}")
print(f"New size: {agg.rio.width} × {agg.rio.height}")

Original size: 146172 × 55183
Original resolution: (0.002462861641117847, -0.002462861641117847)
Done: aggregation complete! Output: ./data/biomass_supply/forest_management_map/2020/plantation_forest_agg20.tif
New size: 7309 × 2760


### Distribute total residue into each grid cell

In [9]:
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling

In [10]:
# -----------------------------------------------------
# User inputs
# -----------------------------------------------------
fileA = r"./data/biomass_supply/forest_management_map/2020/plantation_forest_agg20.tif"

fileB = r"./data/forest_supply/avaliable_residue_GJ_per_ha.tif"

output = r"./data/biomass_supply/forest_management_map/2020/forest_residue_map.tif"

resolution_deg = 0.04166666666667
meters_per_degree = 111320

# -----------------------------------------------------
# Read rasters
# -----------------------------------------------------
with rasterio.open(fileA) as srcA, rasterio.open(fileB) as srcB:

    # A: Resample to B grid
    A = np.empty((srcB.height, srcB.width), dtype=np.float32)

    reproject(
        source=rasterio.band(srcA, 1),
        destination=A,
        src_transform=srcA.transform,
        src_crs=srcA.crs,
        dst_transform=srcB.transform,
        dst_crs=srcB.crs,
        resampling=Resampling.nearest
    )

    # B: Read directly
    B = srcB.read(1).astype(np.float32)

    profile = srcB.profile.copy()

    transform = srcB.transform
    height = srcB.height
    width = srcB.width

# -----------------------------------------------------
# Compute latitude-dependent area (ha)
# -----------------------------------------------------
rows = np.arange(height)

_, latitudes = rasterio.transform.xy(
    transform,
    rows,
    np.zeros(height, dtype=int)
)

latitudes = np.array(latitudes)

lat_rad = np.deg2rad(latitudes)

area_m2 = (
    meters_per_degree * resolution_deg
) * (
    meters_per_degree * resolution_deg * np.cos(lat_rad)
)

area_ha = area_m2 / 10000

area_ha_2d = np.repeat(area_ha[:, np.newaxis], width, axis=1)

# -----------------------------------------------------
# Calculate
# -----------------------------------------------------
scale_factor = (250 * 250) / (5000 * 5000)

result = scale_factor * A * B * area_ha_2d

mask = np.isnan(A) | np.isnan(B)
result[mask] = np.nan

# -----------------------------------------------------
# Save
# -----------------------------------------------------
profile.update(
    dtype=rasterio.float32,
    nodata=np.nan,
    compress="lzw"
)

with rasterio.open(output, "w", **profile) as dst:
    dst.write(result.astype(np.float32), 1)

print("Done!")

Done!


### Apply forest rotation treatment

In [11]:
import rasterio
import numpy as np
from rasterio.warp import reproject, Resampling

In [ ]:
residue_path = r"./data/biomass_supply/forest_management_map/2020/forest_residue_map.tif"
climate_path = r"./data/Climate_Zone/IPCC_Climate_Zones_Map_raster/Raster/ipcc_climate_1985-2015.tif"

output_path = r"./data/biomass_supply/forest_management_map/2020/forest_residue_adjusted_by_rotation.tif"

with rasterio.open(residue_path) as res_src, rasterio.open(climate_path) as cli_src:

    residue = res_src.read(1).astype(np.float32)

    # Create empty climate array matching residue grid
    climate_resampled = np.empty(residue.shape, dtype=np.int16)

    # Resample climate to residue grid
    reproject(
        source=rasterio.band(cli_src, 1),
        destination=climate_resampled,
        src_transform=cli_src.transform,
        src_crs=cli_src.crs,
        dst_transform=res_src.transform,
        dst_crs=res_src.crs,
        resampling=Resampling.nearest   # ⚠️ Categorical data must use nearest neighbor
    )

    # --------- Conditional computation ----------
    conditions = [
        (climate_resampled >= 1) & (climate_resampled <= 4),
        (climate_resampled >= 5) & (climate_resampled <= 8),
        (climate_resampled >= 9) & (climate_resampled <= 12),
    ]

    choices = [
        residue / 7,
        residue / 25,
        residue / 70,
    ]

    new_residue = np.select(conditions, choices, default=0).astype(np.float32)

    # Preserve NoData
    if res_src.nodata is not None:
        new_residue[residue == res_src.nodata] = res_src.nodata

    # Save
    profile = res_src.profile
    profile.update(dtype=rasterio.float32)

    with rasterio.open(output_path, "w", **profile) as dst:
        dst.write(new_residue, 1)

print("Done! Output raster fully inherits the spatial information of residue.")

### Merge agriculture and forest layers

In [ ]:
import rasterio

fileA = r"./data/biomass_supply/forest_management_map/2020/forest_residue_adjusted_by_rotation.tif"
fileB = r"./data/biomass_supply/agricultural_theoretical_energy/overall_biomass_ag_half_SCR_global.tif"

with rasterio.open(fileA) as srcA, rasterio.open(fileB) as srcB:
    print("CRS same:", srcA.crs == srcB.crs)
    print("Resolution same:", srcA.res == srcB.res)
    print("Transform same:", srcA.transform == srcB.transform)
    print("Bounds A:", srcA.bounds)
    print("Bounds B:", srcB.bounds)

In [ ]:
import rioxarray
import numpy as np
from rasterio.transform import from_origin

# =========================================================
# Input file paths
# =========================================================
fileA = r"./data/biomass_supply/forest_management_map/2020/forest_residue_adjusted_by_rotation.tif"
fileB = r"./data/biomass_supply/agricultural_theoretical_energy/overall_biomass_ag_half_SCR_global.tif"
output_file = r"./data/biomass_supply/merged_agri_forest_biomass_sum_global_5km.tif"

# =========================================================
# Read rasters
# =========================================================
A = rioxarray.open_rasterio(fileA)
B = rioxarray.open_rasterio(fileB)

# =========================================================
# Define global extent and resolution (WGS84)
# =========================================================
res = 0.04166666666667   # ≈ 5 km
xmin, ymin, xmax, ymax = -180, -90, 180, 90  # Global lat-lon extent

# Calculate number of rows and columns
new_width = int(np.ceil((xmax - xmin) / res))
new_height = int(np.ceil((ymax - ymin) / res))

# Define geographic transform
transform = from_origin(xmin, ymax, res, res)

# =========================================================
# Reproject and align to a common grid
# =========================================================
A_resampled = A.rio.reproject(
    dst_crs="EPSG:4326",
    transform=transform,
    shape=(new_height, new_width),
    resampling=Resampling.nearest
)

B_resampled = B.rio.reproject(
    dst_crs="EPSG:4326",
    transform=transform,
    shape=(new_height, new_width),
    resampling=Resampling.nearest
)

# =========================================================
# Fill missing values and compute merged result
# =========================================================
A_resampled = A_resampled.fillna(0)
B_resampled = B_resampled.fillna(0)
sum_data = A_resampled/20 + B_resampled # assume 20-year plantation-to-harvest cycle for forest

# Set NaN regions to 0 (no data)
sum_data = sum_data.fillna(0)
sum_data.rio.write_nodata(0, inplace=True)

# =========================================================
# Output result
# =========================================================
sum_data.rio.to_raster(
    output_file,
    dtype="float32"
)

print("Done: merge complete! Output file:", output_file)
print("  Resolution: 0.0416666667° (~5 km)")
print("  Extent: global (-180° to 180°, -90° to 90°)")

### Aggregate results to 40 km resolution

#### First, reproject merged_agri_forest_biomass_sum_global_5km.tif to 5 km Equal Earth projection

In [ ]:
from rasterio.warp import (
    calculate_default_transform,
    reproject,
    Resampling
)
from rasterio.transform import Affine
from pyproj import CRS


# ============================
# File paths
# ============================

input_file = r"./data/biomass_supply/merged_agri_forest_biomass_sum_global_5km.tif"

output_file = r"./data/biomass_supply/merged_biomass_equalearth_5km.tif"


# ============================
# Parameters
# ============================

target_crs = "EPSG:8857"   # Equal Earth

target_resolution = 5000   # meter


# ============================
# Step 1
# Read original data
# ============================

with rasterio.open(input_file) as src:

    biomass = src.read(1).astype("float64")

    src_transform = src.transform
    src_crs = src.crs

    src_profile = src.profile

    nodata = src.nodata


print("Original CRS:", src_crs)
print("Original resolution:", src_transform.a, src_transform.e)



# ============================
# Step 2
# Calculate original pixel area
# EPSG:4326
# ============================

height, width = biomass.shape


# Latitude centers
rows = np.arange(height)

lat = (
    src_transform.f +
    (rows + 0.5) * src_transform.e
)

lat_rad = np.deg2rad(lat)


# Earth radius
R = 6371007.181


# Width in longitude direction
d_lon = abs(src_transform.a)

# Height in latitude direction
d_lat = abs(src_transform.e)


# m²
pixel_area_lat = (
    (np.pi/180*R*d_lat)
    *
    (np.pi/180*R*d_lon*np.cos(lat_rad))
)


# Expand to 2D
pixel_area = np.repeat(
    pixel_area_lat[:, np.newaxis],
    width,
    axis=1
)


# ============================
# Step 3
# GJ/grid -> GJ/m²
# ============================

density = biomass / pixel_area


if nodata is not None:
    density[biomass == nodata] = np.nan



# ============================
# Step 4
# Create Equal Earth 5 km grid
# ============================


with rasterio.open(input_file) as src:

    transform, width_new, height_new = calculate_default_transform(
        src.crs,
        target_crs,
        src.width,
        src.height,
        *src.bounds,
        resolution=target_resolution
    )


print(
    "New size:",
    height_new,
    width_new
)


density_5km = np.empty(
    (height_new, width_new),
    dtype="float64"
)


# ============================
# Step 5
# Reproject density
# ============================

reproject(
    source=density,
    destination=density_5km,

    src_transform=src_transform,
    src_crs=src_crs,

    dst_transform=transform,
    dst_crs=target_crs,

    resampling=Resampling.average
)



# ============================
# Step 6
# GJ/m² -> GJ/grid
# ============================

pixel_area_equalearth = (
    target_resolution *
    target_resolution
)

biomass_5km = (
    density_5km *
    pixel_area_equalearth
)


# ============================
# Step 7
# Save
# ============================

profile = src_profile.copy()

profile.update(
    driver="GTiff",
    height=height_new,
    width=width_new,
    count=1,
    dtype="float32",
    crs=target_crs,
    transform=transform,
    compress="lzw",
    nodata=-9999
)


biomass_5km = np.where(
    np.isnan(biomass_5km),
    -9999,
    biomass_5km
)


with rasterio.open(output_file, "w", **profile) as dst:
    dst.write(
        biomass_5km.astype("float32"),
        1
    )


print("Finished!")
print(output_file)



# ============================
# Step 8
# Global total check
# ============================

original_total = np.nansum(biomass)

new_total = np.sum(
    biomass_5km[biomass_5km!=-9999]
)


print("===================")
print("Original GJ:", original_total)
print("New GJ:", new_total)

print(
    "Relative difference:",
    (new_total-original_total)
    /
    original_total
)

#### Aggregate to 40 km

In [ ]:
import rasterio
import numpy as np
from rasterio.transform import Affine


# ======================================================
# Input/output
# ======================================================

input_file = r"./data/biomass_supply/merged_biomass_equalearth_5km.tif"

output_file = r"./data/biomass_supply/merged_biomass_equalearth_40km.tif"


factor = 8   # 5 km -> 40 km



# ======================================================
# Read 5 km raster
# ======================================================

with rasterio.open(input_file) as src:

    data = src.read(1).astype("float64")

    profile = src.profile.copy()

    transform = src.transform

    crs = src.crs

    nodata = src.nodata


print("Input CRS:", crs)
print("Input resolution:", transform.a, abs(transform.e))
print("Input NoData:", nodata)



# ======================================================
# NoData handling
# ======================================================

if nodata is not None:
    data[data == nodata] = np.nan



# ======================================================
# Trim to integer multiple of factor
# ======================================================

height, width = data.shape

new_height = (height // factor) * factor
new_width = (width // factor) * factor


if height != new_height or width != new_width:
    print(
        f"Crop raster: {height}x{width} -> {new_height}x{new_width}"
    )


data = data[:new_height, :new_width]



# ======================================================
# 8×8 block aggregation
# ======================================================

blocks = data.reshape(
    new_height // factor,
    factor,
    new_width // factor,
    factor
)


# Number of valid 5 km cells in each 40 km block
valid_count = np.sum(
    ~np.isnan(blocks),
    axis=(1,3)
)


# Sum
data_40km = np.nansum(
    blocks,
    axis=(1,3)
)


# If entire 40 km block is NoData
# Keep as NoData
data_40km[valid_count == 0] = np.nan



# ======================================================
# Update 40 km transform
# ======================================================

new_transform = Affine(
    transform.a * factor,
    transform.b,
    transform.c,

    transform.d,
    transform.e * factor,
    transform.f
)


print("Output resolution:",
      new_transform.a,
      abs(new_transform.e))



# ======================================================
# Update metadata
# ======================================================

profile.update(

    height=data_40km.shape[0],

    width=data_40km.shape[1],

    transform=new_transform,

    dtype="float32",

    compress="lzw",

    nodata=-9999

)



# ======================================================
# Save
# ======================================================

data_40km_output = np.where(
    np.isnan(data_40km),
    -9999,
    data_40km
)


with rasterio.open(
    output_file,
    "w",
    **profile
) as dst:

    dst.write(
        data_40km_output.astype("float32"),
        1
    )


print("Saved:")
print(output_file)



# ======================================================
# Global total check
# ======================================================

old_total = np.nansum(data)

new_total = np.sum(
    data_40km_output[data_40km_output != -9999]
)


relative_error = (
    (new_total - old_total)
    /
    old_total
)


print("==============================")
print(f"5 km total : {old_total:,.3f} GJ")
print(f"40 km total: {new_total:,.3f} GJ")
print(f"Difference : {new_total-old_total:,.6f} GJ")
print(f"Relative error: {relative_error:.3e}")
print("==============================")


print("Done: Finished!")